# 04 — Stacking Meta-Modelo (Logistic Regression)

Meta-modelo de stacking que combina las predicciones de los tres modelos base.

## Flujo

```
data/meta/test_*  +  train_processed.csv
        │
        ▼
  LogisticRegression (entrenamiento)
        │
        ├──► submissions/test/  (Pippin_01.01 + 01.03 + 02.00)
        │          └──► submissions/test/Pippin_04.00.csv
        │
        └──► submissions/       (Pippin_01.01 + 01.03 + 02.00)
                   └──► submissions/Pippin_04.00.csv
```

## Mapeo de archivos

| Pippin file   | Feature equivalente   | Transformación       |
|---------------|-----------------------|----------------------|
| Pippin_01.01  | pred_sent_lgbm        | log(predicted_price) |
| Pippin_01.03  | pred_lgbm             | log(predicted_price) |
| Pippin_02.00  | pred_reg_log          | log(predicted_price) |

## 0. Imports y rutas

In [ ]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ── Rutas base (relativas a la raíz del repo) ──────────────────────────────
BASE_DIR  = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
META_DIR  = os.path.join(BASE_DIR, 'data', 'meta') + os.sep
TAB_DIR   = os.path.join(BASE_DIR, 'data', 'tabular') + os.sep
STEST_DIR = os.path.join(BASE_DIR, 'submissions', 'test') + os.sep
SUB_DIR   = os.path.join(BASE_DIR, 'submissions') + os.sep

print('BASE_DIR :', BASE_DIR)
print('META_DIR :', META_DIR)
print('STEST_DIR:', STEST_DIR)
print('SUB_DIR  :', SUB_DIR)

## 1. Cargar predicciones OOF — `data/meta/test_*`

Estas son las predicciones en escala logarítmica que los modelos base generaron
sobre el split de validación interno. Son las features del meta-modelo.

In [ ]:
test_sent = (
    pd.read_csv(META_DIR + 'test_sent_lgbm.csv')
    .rename(columns={'lgbm_pred': 'pred_sent_lgbm'})
)
test_lgbm = (
    pd.read_csv(META_DIR + 'test_lgbm.csv')
    .rename(columns={'lgbm_pred': 'pred_lgbm'})
)
test_reg = (
    pd.read_csv(META_DIR + 'test_regression.csv')
    [['zpid', 'regression_pred_log']]
    .rename(columns={'regression_pred_log': 'pred_reg_log'})
)

X_meta_df = test_sent.merge(test_lgbm, on='zpid').merge(test_reg, on='zpid')
print('Shape features meta (OOF):', X_meta_df.shape)
X_meta_df.head(3)

## 2. Obtener target real (`log_price`)

In [ ]:
train_labels = pd.read_csv(TAB_DIR + 'train_processed.csv', usecols=['zpid', 'log_price'])

df_train = X_meta_df.merge(train_labels, on='zpid', how='inner')
print(f'Filas de entrenamiento del meta-modelo: {len(df_train)}')

X     = df_train[['pred_sent_lgbm', 'pred_lgbm', 'pred_reg_log']].values
y_log = df_train['log_price'].values

print(f'log_price — min: {y_log.min():.3f}  max: {y_log.max():.3f}  mean: {y_log.mean():.3f}')

## 3. Discretizar `log_price` en 20 bins de cuantil

In [ ]:
N_BINS = 20

y_binned, bin_edges = pd.qcut(pd.Series(y_log), q=N_BINS, labels=False, retbins=True)
y_binned = y_binned.values.astype(int)

# Centro de cada bin = media de log_price dentro del bin
bin_centers = np.array([y_log[y_binned == i].mean() for i in range(N_BINS)])

pd.DataFrame({
    'bin'             : range(N_BINS),
    'log_price_center': bin_centers.round(4),
    'precio_aprox'    : np.exp(bin_centers).round(0).astype(int),
    'n_muestras'      : [(y_binned == i).sum() for i in range(N_BINS)]
})

## 4. Entrenar LogisticRegression (sin Optuna)

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(
        multi_class='multinomial',
        solver='lbfgs',
        max_iter=2000,
        C=1.0,
        random_state=42
    ))
])

pipe.fit(X, y_binned)
print('Entrenamiento completado.')

## 5. Evaluación en el conjunto de entrenamiento

In [ ]:
# Predicción continua: esperanza ponderada  E[log_price] = proba @ bin_centers
proba_tr    = pipe.predict_proba(X)
log_pred_tr = proba_tr @ bin_centers

print(f"Accuracy bins  (train): {(pipe.predict(X) == y_binned).mean():.4f}")
print(f"MAE  log_price (train): {mean_absolute_error(y_log, log_pred_tr):.4f}")
print(f"RMSE log_price (train): {np.sqrt(mean_squared_error(y_log, log_pred_tr)):.4f}")
print(f"MAE  precio    (train): ${mean_absolute_error(np.exp(y_log), np.exp(log_pred_tr)):,.0f}")

## 6. Helper: construir features desde archivos Pippin

Los archivos Pippin contienen `predicted_price` (escala real).  
Convertimos a log-escala para que sean comparables con los features de entrenamiento:

```
log(predicted_price) ≈ lgbm_pred  [diferencia < 1e-5]
```

In [ ]:
def build_features_from_pippin(folder):
    """
    Lee Pippin_01.01, Pippin_01.03, Pippin_02.00 de la carpeta indicada.
    Convierte predicted_price -> log(predicted_price) como features del meta-modelo.
    
    Mapeo:
        Pippin_01.01  =>  pred_sent_lgbm
        Pippin_01.03  =>  pred_lgbm
        Pippin_02.00  =>  pred_reg_log
    
    Returns: (zpid Series, X ndarray)
    """
    p01 = pd.read_csv(folder + 'Pippin_01.01.csv').rename(columns={'predicted_price': 'pred_sent_lgbm'})
    p03 = pd.read_csv(folder + 'Pippin_01.03.csv').rename(columns={'predicted_price': 'pred_lgbm'})
    p02 = pd.read_csv(folder + 'Pippin_02.00.csv').rename(columns={'predicted_price': 'pred_reg_log'})

    df = p01.merge(p03, on='zpid').merge(p02, on='zpid')

    # Convertir a log-escala
    for col in ['pred_sent_lgbm', 'pred_lgbm', 'pred_reg_log']:
        df[col] = np.log(df[col])

    X_out = df[['pred_sent_lgbm', 'pred_lgbm', 'pred_reg_log']].values
    return df['zpid'], X_out

print('Helper definido.')

## 7. Predicción A — `submissions/test/` → `Pippin_04.00.csv`

In [ ]:
zpid_test, X_test = build_features_from_pippin(STEST_DIR)

log_pred_a   = pipe.predict_proba(X_test) @ bin_centers
price_pred_a = np.exp(log_pred_a)

out_a = pd.DataFrame({
    'zpid'            : zpid_test.values,
    'predicted_price' : np.round(price_pred_a, 2)
})

out_a.to_csv(STEST_DIR + 'Pippin_04.00.csv', index=False)
print(f'Guardado: {STEST_DIR}Pippin_04.00.csv  ({len(out_a)} filas)')
out_a.describe()

## 8. Predicción B — `submissions/` → `Pippin_04.00.csv`

In [ ]:
zpid_comp, X_comp = build_features_from_pippin(SUB_DIR)

log_pred_b   = pipe.predict_proba(X_comp) @ bin_centers
price_pred_b = np.exp(log_pred_b)

out_b = pd.DataFrame({
    'zpid'            : zpid_comp.values,
    'predicted_price' : np.round(price_pred_b, 2)
})

out_b.to_csv(SUB_DIR + 'Pippin_04.00.csv', index=False)
print(f'Guardado: {SUB_DIR}Pippin_04.00.csv  ({len(out_b)} filas)')
out_b.describe()

## 9. Comparación con modelos base (submissions/)

In [ ]:
p01 = pd.read_csv(SUB_DIR + 'Pippin_01.01.csv')
p03 = pd.read_csv(SUB_DIR + 'Pippin_01.03.csv')
p02 = pd.read_csv(SUB_DIR + 'Pippin_02.00.csv')

cmp = p01[['zpid']].copy()
cmp['sent_lgbm (01.01)'] = p01['predicted_price'].round(0).astype(int)
cmp['lgbm     (01.03)']  = p03['predicted_price'].round(0).astype(int)
cmp['regression (02)']   = p02['predicted_price'].round(0).astype(int)
cmp['META_LR   (04)']    = out_b['predicted_price'].round(0).astype(int)

cmp.head(10)